# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ayesha-Shahzadkhan/flyrank-assignment1/blob/main/work/notebooks/capstone.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

Research Question: Can a logistic regression model, trained on structured content signals, reliably predict which high-visibility pages are at risk of performance decline — and does it outperform a simple rule-based baseline (impressions + staleness thresholds) at flagging pages for refresh?

Decision it supports: Which pages in a content library should be prioritized for refresh vs. left alone — turning raw visibility/staleness data into a ranked action list (REFRESH / REVIEW / NO_ACTION).

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

Primary dataset: content_refresh_anonymized.csv — FlyRank ML Internship dataset, 30,000 rows x 44 columns. Anonymized content-performance data used for the decline-prediction model (Logistic Regression) and its rule-based baseline.

Also explored: FlyRank Warehouse Dataset (Hugging Face, gated, full production release) during Weeks 3–4 for exploratory aggregation and feature exploration at scale — not used for the final trained model in this paper.

What was excluded and why: trend_direction and trend_pct were excluded from model features — these directly encode the outcome being predicted (decline), so including them would cause label leakage. They were retained only to construct the verification-only decline label.

Public-safe note: no client names, URLs, or raw private queries are present in the dataset used; all fields are anonymized.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

Assumptions: A page is a candidate for decline if it has high recent visibility (impressions_90d) but hasn't been updated in a while (days_since_last_update) — the same logic used in the rule-based baseline.

Features: impressions_90d, days_since_last_update — the identical two signals used by the baseline, so the model and baseline are compared on equal footing.

Label definition: Decline label was verified using trend_direction and trend_pct, then those columns (along with impressions_last_30d, clicks_last_30d, sessions_last_30d, impressions_prev_30d, clicks_prev_30d, sessions_prev_30d) were dropped before training — they encode the outcome directly and inflated accuracy artificially, so keeping them would be label leakage.

Baseline: Rule-based — impressions_90d > 81 AND days_since_last_update >= 51 → HIGH_VISIBILITY_STALE → REFRESH; other combinations map to REVIEW/NO_ACTION.

Model: Logistic Regression, trained on the same two features as the baseline.

Validation design: 80/20 train/test split (random split — not time-aware or grouped).

Leakage checks: Identified and removed the trend/window columns above after observing unrealistically high accuracy — a direct sign the model was seeing outcome-correlated data instead of independent signal.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

Random 80/20 split:

Method	Accuracy	Precision	Recall	F1
Baseline (rule-based)	0.599	0.594	0.822	0.690
Logistic Regression	0.644	0.657	0.657	0.686

Logistic Regression showed higher accuracy and precision than the rule-based baseline on this dataset under a random split, while the baseline had noticeably higher recall (0.822 vs 0.657) — meaning the baseline catches more true decline cases, at the cost of more false alarms.

Grouped split (fairer test — grouping removes leakage from related rows landing in both train and test):

Split variant	Base rate	Accuracy	Precision	Recall	F1
With suspect features	0.511	0.583	0.592	0.589	0.590
Without suspect features	0.511	0.578	0.588	0.583	0.585

Once evaluated under a grouped split, performance dropped close to the ~0.51 base rate for both variants — a gap of roughly 7–8 points above chance, not the wide margin the random split suggested. This is a directional result, not a settled one: it's based on one dataset and one split design. Neither the model nor the baseline is ready to make automatic decisions — both work better as a way to flag content for manual review, not as an autonomous call.

## 5. Limitations

*What this work cannot claim.*

Data artifact: ~80% of rows share only 3 distinct staleness (days_since_last_update) values — likely an artifact of how the dataset was generated/anonymized, not real-world content behavior. This limits how much staleness alone can be trusted as a signal.
Split sensitivity: Performance looked strong under a random 80/20 split but dropped close to the base rate under a grouped split — a reminder that random splits can overstate real-world performance when rows aren't independent.
Single dataset, single run: All results come from one anonymized dataset (content_refresh_anonymized.csv, 30K rows) and one train/test configuration. No cross-validation across multiple splits or datasets was done, so these numbers are directional, not definitive.
Two-feature model: Both baseline and model rely only on impressions_90d and days_since_last_update — richer signals (content type, topic, historical update patterns) were not explored, so there's likely unused predictive signal left on the table.
No causal claim: This work identifies association between visibility/staleness and decline — it does not claim refreshing a page causes improved performance, nor does it reflect or reverse-engineer any search engine's ranking algorithm.
Decision-support only: Given the modest, split-sensitive lift over the baseline, this should inform a human reviewer's priority queue — not drive automatic refresh actions.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.